# task 1: car price prediction with linear regression

dataset: https://www.kaggle.com/datasets/boiniabhiram/used-car-price-prediction-dataset-india

set `path` to the downloaded csv file.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

path = 'used_cars.csv'
df = pd.read_csv(path)
df.head()

## eda

In [ ]:
print(df.shape)
df.info()
df.describe()

In [ ]:
print(df.isnull().sum())
print('duplicates:', df.duplicated().sum())

In [ ]:
target = [c for c in df.columns if c.lower() == 'price']
if len(target) == 0:
    target = [c for c in df.columns if 'price' in c.lower()]
target = target[0]
print('target:', target)

## data wrangling

In [ ]:
df = df.drop_duplicates()
df = df.loc[:, df.isnull().mean() < 0.5]
df = df.drop(columns=[c for c in df.columns if c.lower() in ['unnamed: 0', 'id', 'index', 's.no.']])

for c in df.select_dtypes(exclude='number').columns:
    num = pd.to_numeric(df[c].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
    if num.notnull().mean() > 0.8:
        df[c] = num

for c in df.select_dtypes(exclude='number').columns:
    if df[c].nunique() > 50:
        df[c] = df[c].astype(str).str.split().str[0]
    if df[c].nunique() > 50:
        df = df.drop(columns=c)

df = df.dropna(subset=[target])
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].fillna(df[c].median())
    else:
        df[c] = df[c].fillna(df[c].mode()[0])

print(df.shape)
print(df.isnull().sum().sum())

In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(df[target], kde=True)
plt.title('price distribution')
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes('number').corr(), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
x = pd.get_dummies(df.drop(columns=target), drop_first=True).astype(float)
y = df[target]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape, x_test.shape)

## linear regression

In [ ]:
model = LinearRegression()
model.fit(x_train, y_train)
pred_train = model.predict(x_train)
pred_test = model.predict(x_test)
print('intercept:', model.intercept_)

## evaluation

In [ ]:
def report(name, y_true, y_pred):
    print(name)
    print('mae :', round(mean_absolute_error(y_true, y_pred), 4))
    print('mse :', round(mean_squared_error(y_true, y_pred), 4))
    print('rmse:', round(np.sqrt(mean_squared_error(y_true, y_pred)), 4))
    print('r2  :', round(r2_score(y_true, y_pred), 4))

report('train', y_train, pred_train)
report('test', y_test, pred_test)

## plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, yt, yp, name in [(ax[0], y_train, pred_train, 'training'), (ax[1], y_test, pred_test, 'testing')]:
    a.scatter(yt, yp, alpha=0.4)
    a.plot([yt.min(), yt.max()], [yt.min(), yt.max()], color='red')
    a.set_xlabel('actual price')
    a.set_ylabel('predicted price')
    a.set_title(name)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(y_test - pred_test, kde=True)
plt.title('test residuals')
plt.show()